# Analisis Kualitas Data dan Rekonsiliasi Database (DS-1)

Notebook ini memeriksa dataset penjualan Arloraka Coffee, membandingkan seluruh laporan dengan laporan disetujui, dan menyiapkan rincian produk untuk pemetaan database. Kamus data berada di bagian akhir.

Data dibuat dengan `DataDummyCafe.ipynb` untuk pengembangan dan demonstrasi prototipe. Seluruh angka dalam notebook ini adalah hasil simulasi. Dataset sumber tetap disimpan apa adanya; perhitungan dan transformasi memakai salinan di memori.

Jalankan cell dari atas ke bawah. Kebutuhan: Python, pandas, dan numpy. Tambahkan openpyxl jika memakai sumber Excel. Secara bawaan, notebook mencari CSV di folder `dummy-data` pada proyek. Jika memakai Colab, unggah dataset ke `/content` atau isi `DATA_FILE` dengan lokasi berkas.

In [1]:
from pathlib import Path
from numbers import Integral

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Isi dengan Path("lokasi/berkas.csv") untuk memilih sumber secara manual.
DATA_FILE = None
DATA_NAME = "dummy_sales_arloraka_coffee_750"

if DATA_FILE is None:
    base_dirs = [Path.cwd(), Path.cwd().parent, Path("/content")]
    candidates = [
        base / folder / f"{DATA_NAME}{suffix}"
        for suffix in (".csv", ".xlsx")
        for base in base_dirs
        for folder in ("dummy-data", "")
    ]
    file_path = next((p for p in candidates if p.is_file()), None)
    if file_path is None:
        raise FileNotFoundError("Dataset belum ditemukan. Isi DATA_FILE dengan lokasi CSV atau Excel.")
else:
    file_path = Path(DATA_FILE)
    if not file_path.is_file():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {file_path}")

if file_path.suffix.lower() == ".csv":
    df_raw = pd.read_csv(file_path, encoding="utf-8-sig")
elif file_path.suffix.lower() == ".xlsx":
    df_raw = pd.read_excel(file_path)
else:
    raise ValueError("Gunakan berkas .csv atau .xlsx.")

source_shape = df_raw.shape
print(f"Sumber: {file_path.name}")
print(f"Data sumber: {source_shape[0]:,} baris dan {source_shape[1]} kolom.")
display(df_raw.head())

Sumber: dummy_sales_arloraka_coffee_750.csv
Data sumber: 750 baris dan 18 kolom.


,transaction_id,tanggal_jual,freelancer_id,nama_freelancer,americano_qty,latte_qty,aren_latte_qty,matcha_latte_qty,redvelvet_latte_qty,mocha_latte_qty,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen,metode_pembayaran,lokasi_penjualan,status_verifikasi,keterangan
0,TRX-0161,2026-07-01,FL-010,Jihan,0,0,2,0,0,1,3,46000,4600,41400,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD
1,TRX-0465,2026-07-01,FL-007,Gita,1,1,0,0,0,0,2,23000,2300,20700,QRIS,Acara Organisasi,Pending,-
2,TRX-0662,2026-07-01,FL-004,Dimas,0,1,1,0,1,0,3,43000,4300,38700,Cash,Area Perkantoran,Verified,-
3,TRX-0103,2026-07-02,FL-001,Andi,1,0,0,0,0,0,1,10000,1000,9000,QRIS,Area Perkantoran,Verified,-
4,TRX-0251,2026-07-02,FL-004,Dimas,0,1,0,1,0,0,2,29000,2900,26100,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD


## Harga dan aturan dataset

Harga berikut mengikuti `MENU_PRICES` pada generator awal. Tabel ini menjadi satu sumber harga untuk audit omzet dan pembentukan rincian produk. Harga transaksi direkonstruksi dengan asumsi harga generator tetap sepanjang periode simulasi. Untuk data operasional, aplikasi perlu menyimpan harga pada tiap rincian ketika transaksi dibuat.

`product_id` di bawah adalah usulan ID untuk pemetaan awal. Tim perlu menyamakannya dengan ID database sebelum import. Perubahan harga master nantinya tidak boleh mengubah harga laporan lama.

In [2]:
products_master = pd.DataFrame([
    ("PRD-001", "americano_qty", "Americano", 10000),
    ("PRD-002", "latte_qty", "Latte", 13000),
    ("PRD-003", "aren_latte_qty", "Aren Latte", 16000),
    ("PRD-004", "matcha_latte_qty", "Matcha Latte", 16000),
    ("PRD-005", "redvelvet_latte_qty", "Redvelvet Latte", 14000),
    ("PRD-006", "mocha_latte_qty", "Mocha Latte", 14000),
], columns=["product_id", "qty_column", "product_name", "historical_price"])

menu_qty_columns = products_master["qty_column"].tolist()
price_by_qty = products_master.set_index("qty_column")["historical_price"]
STATUS_MAPPING = {"Verified": "approved", "Pending": "pending", "Rejected": "rejected"}
PAYMENT_METHODS = {"Cash", "QRIS", "Transfer"}
LOCATIONS = {
    "Area Kampus", "Event Komunitas", "Area Perkantoran",
    "Car Free Day", "Acara Organisasi", "Penjualan Langsung",
}
FREELANCER_NAMES = dict(zip(
    [f"FL-{i:03d}" for i in range(1, 11)],
    ["Andi", "Bima", "Citra", "Dimas", "Eka", "Farhan", "Gita", "Hana", "Iqbal", "Jihan"],
))
START_DATE = pd.Timestamp("2026-07-01")
END_DATE = pd.Timestamp("2026-10-20")

expected_columns = [
    "transaction_id", "tanggal_jual", "freelancer_id", "nama_freelancer",
    *menu_qty_columns, "total_botol", "omzet_penjualan", "komisi_freelancer",
    "pendapatan_manajemen", "metode_pembayaran", "lokasi_penjualan",
    "status_verifikasi", "keterangan",
]
missing_columns = sorted(set(expected_columns) - set(df_raw.columns))
extra_columns = sorted(set(df_raw.columns) - set(expected_columns))
assert not missing_columns, f"Kolom wajib belum tersedia: {missing_columns}"
assert not extra_columns, f"Kolom di luar skema sumber: {extra_columns}. Periksa skemanya terlebih dahulu."
display(products_master)

,product_id,qty_column,product_name,historical_price
0,PRD-001,americano_qty,Americano,10000
1,PRD-002,latte_qty,Latte,13000
2,PRD-003,aren_latte_qty,Aren Latte,16000
3,PRD-004,matcha_latte_qty,Matcha Latte,16000
4,PRD-005,redvelvet_latte_qty,Redvelvet Latte,14000
5,PRD-006,mocha_latte_qty,Mocha Latte,14000


## Laporan pemeriksaan kualitas data

Pemeriksaan berikut menghitung masalah dari data sumber sebelum transformasi. Kolom `keterangan` boleh kosong. ID, tanggal, kuantitas, nominal, pembayaran, lokasi, dan status wajib terisi. Daftar pemeriksaan menampilkan jumlah masalah serta baris yang perlu diperiksa jika ada temuan.

Sesuai bagian Data Science pada Playbook halaman 13–14, keputusan cleaning mengikuti hasil assessment. Jika seluruh pemeriksaan di bawah lulus, data dapat dilanjutkan ke transformasi tanpa menghapus atau mengisi nilai sumber. Hasil ini berlaku untuk pemeriksaan yang dijalankan dan belum membuktikan alur aplikasi atau import database.

In [3]:
numeric_columns = menu_qty_columns + [
    "total_botol", "omzet_penjualan", "komisi_freelancer", "pendapatan_manajemen",
]
required_non_null = [c for c in expected_columns if c != "keterangan"]
numeric_data = df_raw[numeric_columns].apply(pd.to_numeric, errors="coerce")
parsed_dates = pd.to_datetime(df_raw["tanggal_jual"], errors="coerce")
text_data = df_raw.select_dtypes(include=["object", "string"])
empty_text = text_data.apply(lambda s: s.astype("string").str.strip().eq("").fillna(False))
required_text = [c for c in empty_text.columns if c in required_non_null]

row_checks = {
    "Nilai kosong pada kolom wajib": df_raw[required_non_null].isna().any(axis=1),
    "String kosong pada kolom wajib": empty_text[required_text].any(axis=1),
    "Baris duplikat setelah kemunculan pertama": df_raw.duplicated(),
    "ID transaksi ganda setelah kemunculan pertama": df_raw["transaction_id"].duplicated(),
    "Format ID transaksi tidak valid": ~df_raw["transaction_id"].astype("string").str.fullmatch(r"TRX-[0-9]{4}", na=False),
    "ID freelancer tidak terdaftar pada generator": ~df_raw["freelancer_id"].isin(FREELANCER_NAMES),
    "Nama tidak sesuai ID freelancer": df_raw["nama_freelancer"].ne(df_raw["freelancer_id"].map(FREELANCER_NAMES)),
    "Tanggal tidak valid": parsed_dates.isna(),
    "Tanggal di luar periode generator": parsed_dates.notna() & ~parsed_dates.between(START_DATE, END_DATE),
    "Nilai numerik tidak valid atau tidak terhingga": ~np.isfinite(numeric_data).all(axis=1),
    "Kuantitas atau nominal negatif": numeric_data.lt(0).any(axis=1),
    "Kuantitas atau nominal bukan bilangan bulat": (numeric_data.notna() & numeric_data.mod(1).ne(0)).any(axis=1),
    "Total botol tidak positif": numeric_data["total_botol"].le(0),
    "Metode pembayaran tidak dikenal": ~df_raw["metode_pembayaran"].isin(PAYMENT_METHODS),
    "Lokasi tidak dikenal": ~df_raw["lokasi_penjualan"].isin(LOCATIONS),
    "Status verifikasi tidak dikenal": ~df_raw["status_verifikasi"].isin(STATUS_MAPPING),
}

quality_report = pd.DataFrame([
    {"pemeriksaan": label, "baris_bermasalah": int(mask.sum())}
    for label, mask in row_checks.items()
])
conflicting_names = df_raw.groupby("freelancer_id")["nama_freelancer"].nunique(dropna=False)
conflicting_ids = conflicting_names[conflicting_names.gt(1)].index.tolist()
quality_report.loc[len(quality_report)] = ["ID freelancer dengan nama tidak konsisten", len(conflicting_ids)]

dtype_report = pd.DataFrame({
    "kolom": numeric_columns,
    "tipe_sumber": [str(df_raw[c].dtype) for c in numeric_columns],
    "tipe_numerik": [pd.api.types.is_numeric_dtype(df_raw[c]) for c in numeric_columns],
})
quality_report.loc[len(quality_report)] = ["Kolom numerik terbaca sebagai tipe lain", int((~dtype_report["tipe_numerik"]).sum())]

display(quality_report)
display(dtype_report)
problem_frames = [
    df_raw.loc[mask.fillna(True), ["transaction_id"]].assign(pemeriksaan=label)
    for label, mask in row_checks.items() if mask.fillna(True).any()
]
problem_rows = (
    pd.concat(problem_frames, ignore_index=True)
    if problem_frames else pd.DataFrame(columns=["transaction_id", "pemeriksaan"])
)
if not problem_rows.empty:
    display(problem_rows)
if conflicting_ids:
    print("ID dengan nama tidak konsisten:", conflicting_ids)

print(f"Nilai kosong seluruh sumber: {int(df_raw.isna().sum().sum())} sel.")
print(f"String kosong seluruh sumber: {int(empty_text.sum().sum())} sel.")
assert quality_report["baris_bermasalah"].eq(0).all(), "Ada temuan kualitas data. Tinjau laporan sebelum melanjutkan."

df = df_raw.copy(deep=True)
df["tanggal_jual"] = parsed_dates
df[numeric_columns] = numeric_data.astype("int64")
print(f"Periode: {df.tanggal_jual.min().date()} hingga {df.tanggal_jual.max().date()}.")
print(f"Tanggal dengan laporan: {df.tanggal_jual.nunique()}; freelancer: {df.freelancer_id.nunique()}.")

,pemeriksaan,baris_bermasalah
0,Nilai kosong pada kolom wajib,0
1,String kosong pada kolom wajib,0
2,Baris duplikat setelah kemunculan pertama,0
3,ID transaksi ganda setelah kemunculan pertama,0
4,Format ID transaksi tidak valid,0
5,ID freelancer tidak terdaftar pada generator,0
6,Nama tidak sesuai ID freelancer,0
7,Tanggal tidak valid,0
8,Tanggal di luar periode generator,0
9,Nilai numerik tidak valid atau tidak terhingga,0


,kolom,tipe_sumber,tipe_numerik
0,americano_qty,int64,True
1,latte_qty,int64,True
2,aren_latte_qty,int64,True
3,matcha_latte_qty,int64,True
4,redvelvet_latte_qty,int64,True
5,mocha_latte_qty,int64,True
6,total_botol,int64,True
7,omzet_penjualan,int64,True
8,komisi_freelancer,int64,True
9,pendapatan_manajemen,int64,True


Nilai kosong seluruh sumber: 0 sel.
String kosong seluruh sumber: 0 sel.
Periode: 2026-07-01 hingga 2026-10-20.
Tanggal dengan laporan: 112; freelancer: 10.


In [4]:
arithmetic_check = df[["transaction_id", "total_botol", "omzet_penjualan"]].copy()
arithmetic_check["total_botol_hitung"] = df[menu_qty_columns].sum(axis=1)
arithmetic_check["omzet_hitung"] = df[menu_qty_columns].mul(price_by_qty, axis="columns").sum(axis=1)
arithmetic_check["selisih_botol"] = arithmetic_check["total_botol_hitung"] - df["total_botol"]
arithmetic_check["selisih_omzet"] = arithmetic_check["omzet_hitung"] - df["omzet_penjualan"]
arithmetic_check["selisih_pembagian"] = df["komisi_freelancer"] + df["pendapatan_manajemen"] - df["omzet_penjualan"]
arithmetic_differences = arithmetic_check.loc[
    arithmetic_check[["selisih_botol", "selisih_omzet", "selisih_pembagian"]].ne(0).any(axis=1)
]
display(pd.DataFrame({
    "pemeriksaan": ["Total botol", "Omzet berdasarkan harga generator", "Komisi + bagian manajemen"],
    "laporan_berselisih": [
        int(arithmetic_check[c].ne(0).sum())
        for c in ["selisih_botol", "selisih_omzet", "selisih_pembagian"]
    ],
}))
if not arithmetic_differences.empty:
    display(arithmetic_differences)
assert arithmetic_differences.empty, "Ada selisih aritmetika. Periksa laporan dan harga sumber."
print("Tidak ditemukan masalah pada pemeriksaan kualitas dan aritmetika di atas; cleaning nilai sumber tidak diperlukan.")

,pemeriksaan,laporan_berselisih
0,Total botol,0
1,Omzet berdasarkan harga generator,0
2,Komisi + bagian manajemen,0


Tidak ditemukan masalah pada pemeriksaan kualitas dan aritmetika di atas; cleaning nilai sumber tidak diperlukan.


## Aturan bisnis dan pengakuan komisi

Pemetaan status mengikuti Overview: `Verified → approved`, `Pending → pending`, dan `Rejected → rejected`. Status `revision` digunakan pada alur aplikasi dan belum ada pada dataset awal. Hanya laporan approved yang masuk kontribusi botol dan omzet.

Komisi per laporan adalah 10% omzet, dibulatkan ke atas ke rupiah penuh. Bagian manajemen adalah omzet dikurangi komisi. Kolom `commission_spec` menyimpan hasil hitung aturan tersebut untuk rekonsiliasi; nominal pada baris Pending atau Rejected tidak menjadi hak komisi yang diakui.

Laporan approved pada bulan berjalan menghasilkan estimasi komisi. Pengakuan dilakukan pada akhir bulan kalender berdasarkan bulan tanggal penjualan, dengan menjumlahkan komisi per laporan tanpa pembulatan ulang. Dataset ini hanya memuat status terkini, sehingga tidak membuktikan kapan laporan disetujui atau kapan pengakuan dilakukan. Tabel bulanan di bawah adalah acuan simulasi untuk Athallah (DS-2) dan Faiq.

Pembayaran dicatat terpisah sesuai kesepakatan dengan manajemen. Akhir bulan merupakan waktu pengakuan komisi dalam rancangan proyek; tanggal dan bukti pembayaran belum tersedia pada dataset.

In [5]:
def calculate_commission(omzet):
    '''Komisi 10%, dibulatkan ke atas per laporan; input rupiah bulat nonnegatif.'''
    if isinstance(omzet, (bool, np.bool_)) or not isinstance(omzet, Integral):
        raise TypeError("Omzet harus berupa rupiah bulat.")
    if omzet < 0:
        raise ValueError("Omzet tidak boleh negatif.")
    return (int(omzet) + 9) // 10


df["commission_spec"] = df["omzet_penjualan"].map(calculate_commission)
df["management_spec"] = df["omzet_penjualan"] - df["commission_spec"]
df["status_aplikasi"] = df["status_verifikasi"].map(STATUS_MAPPING)

commission_check = df[["transaction_id", "omzet_penjualan", "komisi_freelancer", "commission_spec"]].copy()
commission_check["selisih_komisi"] = df["commission_spec"] - df["komisi_freelancer"]
commission_check["selisih_manajemen"] = df["management_spec"] - df["pendapatan_manajemen"]
commission_differences = commission_check.loc[
    commission_check[["selisih_komisi", "selisih_manajemen"]].ne(0).any(axis=1)
]
print(f"Laporan dengan selisih aturan komisi: {len(commission_differences)}.")
if not commission_differences.empty:
    display(commission_differences)
assert (df["commission_spec"] + df["management_spec"]).eq(df["omzet_penjualan"]).all()
print(f"Kolom sumber: {source_shape[1]}; kolom kerja setelah tiga kolom turunan: {df.shape[1]}.")

Laporan dengan selisih aturan komisi: 0.
Kolom sumber: 18; kolom kerja setelah tiga kolom turunan: 21.


## Seluruh data dan laporan disetujui

Tabel memakai `commission_spec` dan `management_spec` hasil perhitungan ulang. Kolom seluruh data berfungsi sebagai pembanding audit. Jumlah komisi pada kolom approved adalah dasar rekap estimasi/pengakuan sesuai periodenya; angka tersebut tidak membuktikan pembayaran.

In [6]:
df_verified = df.loc[df["status_aplikasi"].eq("approved")].copy()

def summarize(frame):
    return [
        len(frame), int(frame["total_botol"].sum()), int(frame["omzet_penjualan"].sum()),
        int(frame["commission_spec"].sum()), int(frame["management_spec"].sum()),
    ]


df_comparison = pd.DataFrame({
    "Metrik": [
        "Jumlah laporan", "Total botol", "Omzet (Rp)",
        "Jumlah nominal komisi menurut aturan (Rp)", "Bagian manajemen menurut aturan (Rp)",
    ],
    "Seluruh data (audit)": summarize(df),
    "Hanya approved": summarize(df_verified),
})
display(df_comparison)

status_recap = df.groupby("status_aplikasi", as_index=False).agg(
    jumlah_laporan=("transaction_id", "count"),
    total_botol=("total_botol", "sum"),
    omzet=("omzet_penjualan", "sum"),
    nominal_10_persen=("commission_spec", "sum"),
)
display(status_recap)
assert df_verified["status_verifikasi"].eq("Verified").all()
assert status_recap["jumlah_laporan"].sum() == len(df)

,Metrik,Seluruh data (audit),Hanya approved
0,Jumlah laporan,750,693
1,Total botol,2145,1986
2,Omzet (Rp),30019000,27776000
3,Jumlah nominal komisi menurut aturan (Rp),3001900,2777600
4,Bagian manajemen menurut aturan (Rp),27017100,24998400


,status_aplikasi,jumlah_laporan,total_botol,omzet,nominal_10_persen
0,approved,693,1986,27776000,2777600
1,pending,42,124,1738000,173800
2,rejected,15,35,505000,50500


In [7]:
# Tanggal acuan untuk membaca bulan berjalan pada simulasi ini.
AS_OF_DATE = END_DATE
monthly_commission = (
    df_verified.assign(periode_komisi=df_verified["tanggal_jual"].dt.to_period("M"))
    .groupby("periode_komisi", as_index=False)
    .agg(jumlah_laporan=("transaction_id", "count"), omzet=("omzet_penjualan", "sum"),
         komisi_acuan=("commission_spec", "sum"), bagian_manajemen=("management_spec", "sum"))
)
monthly_commission["akhir_bulan"] = monthly_commission["periode_komisi"].dt.to_timestamp(how="end").dt.normalize()
monthly_commission["keterangan_simulasi"] = np.where(
    monthly_commission["akhir_bulan"].le(AS_OF_DATE),
    "rekap akhir bulan berdasarkan status dataset", "estimasi bulan berjalan",
)
print(f"Tanggal acuan simulasi: {AS_OF_DATE.date()}.")
display(monthly_commission)
assert monthly_commission["komisi_acuan"].sum() == df_verified["commission_spec"].sum()
print("Waktu persetujuan, pengakuan aktual, dan pembayaran belum tersedia pada dataset.")

Tanggal acuan simulasi: 2026-10-20.


,periode_komisi,jumlah_laporan,omzet,komisi_acuan,bagian_manajemen,akhir_bulan,keterangan_simulasi
0,2026-07,187,7374000,737400,6636600,2026-07-31,rekap akhir bulan berdasarkan status dataset
1,2026-08,207,8253000,825300,7427700,2026-08-31,rekap akhir bulan berdasarkan status dataset
2,2026-09,183,7464000,746400,6717600,2026-09-30,rekap akhir bulan berdasarkan status dataset
3,2026-10,116,4685000,468500,4216500,2026-10-31,estimasi bulan berjalan


Waktu persetujuan, pengakuan aktual, dan pembayaran belum tersedia pada dataset.


## Pengujian pembulatan komisi

Harga generator membuat seluruh omzet menjadi kelipatan Rp1.000, sehingga data awal belum memperlihatkan perbedaan `round` dan pembulatan ke atas. Kasus uji berikut memakai nominal rupiah bulat di sekitar batas pembulatan. Fungsi komisi menggunakan aritmetika integer agar hasil tidak bergantung pada presisi floating point.

In [8]:
rounding_cases = [
    (0, 0), (1, 1), (9, 1), (10, 1), (11, 2),
    (15000, 1500), (15001, 1501), (15004, 1501), (15005, 1501),
    (15009, 1501), (15010, 1501), (15011, 1502),
]
rounding_results = []
for omzet, expected in rounding_cases:
    actual = calculate_commission(omzet)
    assert actual == expected, f"Komisi tidak sesuai pada omzet {omzet}."
    management = omzet - actual
    assert actual + management == omzet
    rounding_results.append({
        "omzet": omzet, "round_generator_lama": int(round(omzet * 0.10)),
        "komisi_spesifikasi": actual, "hasil_diharapkan": expected,
    })

for invalid in [-1, 1.5, None, np.nan, True, "15000"]:
    try:
        calculate_commission(invalid)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"Input seharusnya ditolak: {invalid!r}")

display(pd.DataFrame(rounding_results))
print(f"{len(rounding_cases)} kasus pembulatan dan 6 kasus input tidak valid lulus.")

,omzet,round_generator_lama,komisi_spesifikasi,hasil_diharapkan
0,0,0,0,0
1,1,0,1,1
2,9,1,1,1
3,10,1,1,1
4,11,1,2,2
5,15000,1500,1500,1500
6,15001,1500,1501,1501
7,15004,1500,1501,1501
8,15005,1500,1501,1501
9,15009,1501,1501,1501


12 kasus pembulatan dan 6 kasus input tidak valid lulus.


## Rincian produk dan rekonsiliasi laporan

Setiap pasangan laporan dan produk dengan kuantitas positif menjadi satu rincian. Harga berasal dari tabel generator di awal notebook. Rekonsiliasi dilakukan per laporan, sehingga selisih antartransaksi tidak saling menutupi pada total keseluruhan.

Tabel ini masih berada di memori. Pengujian import ulang, koreksi laporan, dan perubahan harga master dilakukan bersama tim aplikasi setelah skema database disepakati.

In [9]:
df_details = (
    df[["transaction_id", *menu_qty_columns]]
    .melt(id_vars="transaction_id", value_vars=menu_qty_columns,
          var_name="qty_column", value_name="quantity")
    .loc[lambda x: x["quantity"].gt(0)]
    .merge(products_master, on="qty_column", how="left", validate="many_to_one")
)
df_details["subtotal"] = df_details["quantity"] * df_details["historical_price"]
df_details = df_details[[
    "transaction_id", "product_id", "product_name", "quantity", "historical_price", "subtotal",
]].sort_values(["transaction_id", "product_id"]).reset_index(drop=True)

assert not df_details[["transaction_id", "product_id"]].duplicated().any()
assert df_details["product_id"].notna().all()
assert df_details["transaction_id"].isin(df["transaction_id"]).all()

detail_totals = df_details.groupby("transaction_id").agg(
    botol_rincian=("quantity", "sum"), omzet_rincian=("subtotal", "sum"),
)
detail_reconciliation = df[["transaction_id", "total_botol", "omzet_penjualan"]].merge(
    detail_totals, on="transaction_id", how="left", validate="one_to_one",
)
detail_reconciliation["selisih_botol"] = detail_reconciliation["botol_rincian"] - detail_reconciliation["total_botol"]
detail_reconciliation["selisih_omzet"] = detail_reconciliation["omzet_rincian"] - detail_reconciliation["omzet_penjualan"]
detail_differences = detail_reconciliation.loc[
    detail_reconciliation[["selisih_botol", "selisih_omzet"]].ne(0).any(axis=1)
]
if not detail_differences.empty:
    display(detail_differences)
assert detail_differences.empty, "Rincian produk belum cocok dengan laporan sumber."
assert len(df_details) == int(df[menu_qty_columns].gt(0).sum().sum())
assert df_details["subtotal"].sum() == df["omzet_penjualan"].sum()
print(f"Rincian produk: {len(df_details):,} baris; laporan diperiksa: {len(detail_reconciliation):,}.")
print(f"Laporan berselisih: {len(detail_differences)}; total rincian: Rp{df_details['subtotal'].sum():,}.")
display(df_details.head(10))

Rincian produk: 1,681 baris; laporan diperiksa: 750.
Laporan berselisih: 0; total rincian: Rp30,019,000.


,transaction_id,product_id,product_name,quantity,historical_price,subtotal
0,TRX-0001,PRD-001,Americano,1,10000,10000
1,TRX-0001,PRD-002,Latte,2,13000,26000
2,TRX-0001,PRD-004,Matcha Latte,1,16000,16000
3,TRX-0002,PRD-002,Latte,3,13000,39000
4,TRX-0002,PRD-003,Aren Latte,1,16000,16000
5,TRX-0003,PRD-003,Aren Latte,2,16000,32000
6,TRX-0004,PRD-001,Americano,1,10000,10000
7,TRX-0004,PRD-002,Latte,1,13000,13000
8,TRX-0004,PRD-006,Mocha Latte,1,14000,14000
9,TRX-0005,PRD-001,Americano,1,10000,10000


## Kamus data

CSV sumber memiliki 18 kolom. Tanggal dibaca dari CSV sebagai teks, lalu diubah menjadi datetime pada salinan kerja. Kolom nominal dan kuantitas menggunakan bilangan bulat. Tanda `-` pada keterangan berarti generator tidak menambahkan catatan khusus.

| Kolom sumber | Arti | Tipe kerja / database | Satuan | Contoh | Validasi | Boleh kosong? |
|---|---|---|---|---|---|---|
| `transaction_id` | Identitas asal satu laporan penjualan | String / VARCHAR | - | `TRX-0161` | Pola `TRX-` + 4 digit; unik pada seed ini | Tidak |
| `tanggal_jual` | Tanggal penjualan dan dasar periode komisi | Datetime / DATE | YYYY-MM-DD | `2026-07-01` | Tanggal valid; pada seed ini 1 Juli–20 Oktober 2026 | Tidak |
| `freelancer_id` | Identitas freelancer pemilik laporan | String / VARCHAR | - | `FL-010` | Terdaftar pada master freelancer | Tidak |
| `nama_freelancer` | Nama tampilan freelancer | String / VARCHAR | - | `Jihan` | Teks berisi; sesuai pasangan ID pada master | Tidak |
| `americano_qty` | Americano terjual | Integer / INTEGER | Botol | `1` | Bilangan bulat ≥ 0 | Tidak |
| `latte_qty` | Latte terjual | Integer / INTEGER | Botol | `2` | Bilangan bulat ≥ 0 | Tidak |
| `aren_latte_qty` | Aren Latte terjual | Integer / INTEGER | Botol | `1` | Bilangan bulat ≥ 0 | Tidak |
| `matcha_latte_qty` | Matcha Latte terjual | Integer / INTEGER | Botol | `0` | Bilangan bulat ≥ 0 | Tidak |
| `redvelvet_latte_qty` | Redvelvet Latte terjual | Integer / INTEGER | Botol | `0` | Bilangan bulat ≥ 0 | Tidak |
| `mocha_latte_qty` | Mocha Latte terjual | Integer / INTEGER | Botol | `1` | Bilangan bulat ≥ 0 | Tidak |
| `total_botol` | Jumlah botol pada laporan | Integer / INTEGER | Botol | `3` | Positif; sama dengan jumlah seluruh `_qty` | Tidak |
| `omzet_penjualan` | Omzet laporan sebelum pembagian komisi | Integer / BIGINT | Rupiah | `46000` | Jumlah kuantitas × harga saat transaksi | Tidak |
| `komisi_freelancer` | Nominal komisi keluaran generator awal, pada semua status | Integer / BIGINT | Rupiah | `4600` | Direkonsiliasi terhadap `commission_spec`; hak komisi mengikuti status dan periode | Tidak |
| `pendapatan_manajemen` | Sisa omzet setelah komisi; biaya usaha belum dikurangi | Integer / BIGINT | Rupiah | `41400` | Omzet dikurangi komisi sumber | Tidak |
| `metode_pembayaran` | Metode pembayaran transaksi penjualan | String / VARCHAR | - | `QRIS` | Cash, QRIS, atau Transfer | Tidak |
| `lokasi_penjualan` | Kategori lokasi penjualan | String / VARCHAR | - | `Area Kampus` | Salah satu dari enam lokasi generator pada cell aturan dataset | Tidak |
| `status_verifikasi` | Status laporan pada dataset awal | String / VARCHAR | - | `Verified` | Verified, Pending, atau Rejected | Tidak |
| `keterangan` | Catatan penjualan sintetis | String / TEXT | - | `Penjualan saat event komunitas` | Teks bebas; tidak membuktikan alasan keputusan verifikasi | Ya |

| Kolom turunan | Arti | Tipe / satuan | Aturan |
|---|---|---|---|
| `commission_spec` | Nominal 10% hasil hitung aturan proyek per laporan | Integer / rupiah | `(omzet_penjualan + 9) // 10` untuk rupiah bulat nonnegatif |
| `management_spec` | Sisa omzet menurut aturan komisi proyek | Integer / rupiah | `omzet_penjualan - commission_spec` |
| `status_aplikasi` | Status untuk pemetaan aplikasi | String | Verified → approved; Pending → pending; Rejected → rejected |

Pada rincian produk, `transaction_id` menghubungkan rincian dengan laporan, `product_id` menghubungkan produk usulan, dan `quantity` menyimpan botol positif. `historical_price` adalah harga generator yang dipakai untuk merekonstruksi seed ini; `subtotal = quantity × historical_price`. Kunci pasangan `(transaction_id, product_id)` tidak boleh ganda.

## Sumber, asumsi, dan batasan

Sumber dataset adalah `DataDummyCafe.ipynb`, dengan seed 42 dan 750 transaksi. Generator mencatat harga menu serta aturan bagi hasil berdasarkan informasi UMKM sebagaimana dijelaskan pada notebook generator. Frekuensi transaksi, identitas freelancer dummy, lokasi, metode pembayaran, dan komposisi pembelian merupakan asumsi simulasi.

Generator memilih total botol dari 1, 2, 3, 4, 5, 6, 8, atau 10, lalu membagikannya ke produk. Kuantitas satu produk dapat melebihi 3. Probabilitas pemilihan menu adalah 14%, 16%, 22%, 20%, 14%, dan 14% sesuai urutan tabel produk. Aktivitas freelancer juga memakai bobot yang ditetapkan generator.

Status diacak dengan probabilitas Verified 92%, Pending 6%, dan Rejected 2%. Status tersebut tidak menjadi bukti kelayakan transaksi aktual atau label pelatihan model verifikasi. Tanggal Jumat diberi bobot 1,2; Sabtu dan Minggu 1,4; hari lain 1,0. Pola akhir pekan dan peringkat produk/freelancer karena itu dipengaruhi rancangan generator.

Oktober hanya mencakup tanggal 1–20. Gunakan rentang tanggal setara atau rata-rata harian saat membandingkannya dengan bulan penuh, dan tetap tulis bahwa datanya simulasi. Hasil ini belum mengukur perubahan omzet, efisiensi operasional, atau perilaku pelanggan nyata.

Harga pada dataset belum menyimpan riwayat perubahan per transaksi. Notebook ini merekonstruksi harga tetap generator; implementasi aplikasi harus mempertahankan snapshot harga pada laporan lama. Akun, bukti, target, waktu keputusan, revisi, koreksi, dan pembayaran memerlukan data pendukung terpisah. Pengujian database dan alur tersebut belum dilakukan dalam notebook ini.

## Perubahan pada assessment

- Menyamakan harga rincian dengan generator awal dan memakai satu tabel produk.
- Menambahkan audit yang dijalankan dari data sumber, daftar selisih, dan pemeriksaan per laporan.
- Memakai komisi hasil hitung ulang pada tabel pembanding serta menguji batas pembulatannya.
- Memisahkan kontribusi approved, estimasi, acuan pengakuan bulanan, dan pembayaran yang belum tercatat.
- Memperbaiki jumlah kolom, lokasi, arti bagian manajemen, dan asumsi kuantitas pada kamus data.

Pekerjaan berikutnya adalah menyepakati ID produk dan pemetaan database bersama Kevin, lalu menguji import ulang tanpa membuat laporan atau rincian ganda.